# **Multimodal AI System for Startup Pitch Evaluation (English–Tamil)**

**Architecture**: Input & Preprocessing → Parallel Feature Extraction (Text/Visual/Audio) → Cross-Attention Fusion → Hierarchical Scoring → Investor Dashboard


## 0 · Install Dependencies

In [1]:
!pip install pydantic --quiet

## 1 · Data Schemas
Pydantic models for all inputs and outputs.

In [2]:
from __future__ import annotations
from pydantic import BaseModel, Field


class PitchVideoInput(BaseModel):
    file_name: str = Field(..., description="Pitch video file name")
    file_format: str = Field(default="mp4")
    duration_sec: int = Field(default=60, ge=5)
    transcript_text: str = Field(default="", description="Optional ASR transcript")


class SlideInput(BaseModel):
    title: str = Field(default="")
    content: str = Field(default="")


class UserDetails(BaseModel):
    founder_name: str = Field(default="")
    startup_name: str = Field(default="")
    sector: str = Field(default="")
    stage: str = Field(default="")


class PitchInput(BaseModel):
    title: str = Field(..., description="Startup name or pitch title")
    transcript: str = Field(default="", description="Full pitch transcript")
    language_hint: str = Field(default="en-ta", description="Language mix hint: en, ta, en-ta")
    presenter_profile: dict = Field(default_factory=dict)
    slide_text: list[str] = Field(default_factory=list)
    video: PitchVideoInput | None = Field(default=None)
    slides: list[SlideInput] = Field(default_factory=list)
    user_details: UserDetails | None = Field(default=None)


class MetricScore(BaseModel):
    name: str
    score: float
    rationale: str


class ChunkReport(BaseModel):
    chunk_id: int
    start_sec: int
    end_sec: int
    text_metrics: list[MetricScore]
    av_metrics: list[MetricScore]
    attention: dict[str, float]
    risk_flags: list[str]
    aggregate_score: float


class EvaluationSummary(BaseModel):
    overall_score: float
    confidence_score: float
    investment_band: str
    language_detected: str
    strengths: list[str]
    weaknesses: list[str]
    suggestions: list[str]


class DashboardSeriesPoint(BaseModel):
    label: str
    value: float


class InvestorDashboard(BaseModel):
    quantitative_scores: list[DashboardSeriesPoint]
    modality_weights: list[DashboardSeriesPoint]
    risk_distribution: list[DashboardSeriesPoint]


class EvaluationResponse(BaseModel):
    request_id: str
    summary: EvaluationSummary
    chunk_reports: list[ChunkReport]
    dashboard: InvestorDashboard


print("Schemas defined")


✅ Schemas defined


## 2 · Feature Encoders
Deterministic encoders for Text, Visual, and Audio modalities.
Swap these with real model inference (Whisper, ViT, Wav2Vec2) when your dataset is ready.

In [3]:
import hashlib
import re


class TextEncoder:
    """
    Text feature extractor with bilingual (English / Tamil) support.
    Detects language, computes keyword signals, and produces a 24-dim embedding.
    Replace `infer()` with a real NLP model (e.g. Google MuRIL / HuggingFace) later.
    """

    def __init__(self, embedding_dim: int = 24):
        self.embedding_dim = embedding_dim

    def _hash_to_vector(self, value: str) -> list[float]:
        digest = hashlib.sha256(value.encode("utf-8")).digest()
        values = [b / 255.0 for b in digest]
        repeats = (self.embedding_dim // len(values)) + 1
        return (values * repeats)[: self.embedding_dim]

    def _detect_language(self, text: str, hint: str) -> str:
        tamil = len(re.findall(r"[\u0B80-\u0BFF]", text))
        english = len(re.findall(r"[A-Za-z]", text))
        if tamil > 0 and english > 0:
            return "ta-en"
        if tamil > 0:
            return "ta"
        if "ta" in hint.lower() and "en" in hint.lower():
            return "ta-en"
        return "en"

    @staticmethod
    def _clamp(v: float) -> float:
        return max(0.0, min(10.0, v))

    def infer(self, chunk_text: str, language_hint: str, slide_context: str) -> dict:
        text = " ".join(chunk_text.replace("\n", " ").split())
        words = text.split()
        unique_ratio = len(set(words)) / max(1, len(words))
        lang = self._detect_language(text, language_hint)
        low = text.lower()

        problem_sig    = 2.0 if "problem" in low else 0.0
        market_sig     = 2.0 if "market" in low else 0.0
        traction_sig   = 2.0 if any(k in low for k in ["pilot", "growth", "revenue"]) else 0.0
        biz_sig        = 2.0 if any(k in low for k in ["price", "subscription", "saas", "margin"]) else 0.0
        team_sig       = 2.0 if any(k in low for k in ["team", "founder", "experience"]) else 0.0
        lang_bonus     = 1.0 if lang == "ta-en" else 0.4

        return {
            "embedding":              self._hash_to_vector(f"text::{text}::{slide_context}"),
            "language_detected":      lang,
            "problem_clarity":        self._clamp(4.0 + problem_sig + unique_ratio * 1.5),
            "market_opportunity":     self._clamp(4.0 + market_sig  + unique_ratio * 1.3),
            "solution_uniqueness":    self._clamp(4.5 + unique_ratio * 3.0),
            "traction_evidence":      self._clamp(3.5 + traction_sig + min(len(words), 40) * 0.08),
            "business_model_strength":self._clamp(3.5 + biz_sig     + min(len(words), 30) * 0.07),
            "team_readiness":         self._clamp(3.0 + team_sig    + lang_bonus),
        }


class VisualEncoder:
    """
    Visual proxy encoder for slide-based delivery signals.
    Replace with VideoMAE / ViT-Base frame inference when video data is available.
    """

    def __init__(self, embedding_dim: int = 24):
        self.embedding_dim = embedding_dim

    def _hash_to_vector(self, value: str) -> list[float]:
        digest = hashlib.sha256(value.encode("utf-8")).digest()
        values = [b / 255.0 for b in digest]
        repeats = (self.embedding_dim // len(values)) + 1
        return (values * repeats)[: self.embedding_dim]

    @staticmethod
    def _clamp(v: float) -> float:
        return max(0.0, min(10.0, v))

    def infer(self, slide_context: str, chunk_id: int, user_stage: str) -> dict:
        text = slide_context or "no-slides"
        density = min(len(text.split()), 120) / 120.0
        stage_bonus = 0.8 if user_stage.lower() in {"seed", "series-a", "series a"} else 0.3
        return {
            "embedding":           self._hash_to_vector(f"visual::{chunk_id}::{text}::{user_stage}"),
            "delivery_clarity":    self._clamp(4.2 + density * 3.8),
            "presenter_confidence":self._clamp(4.0 + (chunk_id % 5) * 0.9 + stage_bonus),
        }


class AudioEncoder:
    """
    Audio proxy encoder for prosody / pace signals.
    Replace with Wav2Vec2 / Facebook Prosody model when audio files are available.
    """

    def __init__(self, embedding_dim: int = 24):
        self.embedding_dim = embedding_dim

    def _hash_to_vector(self, value: str) -> list[float]:
        digest = hashlib.sha256(value.encode("utf-8")).digest()
        values = [b / 255.0 for b in digest]
        repeats = (self.embedding_dim // len(values)) + 1
        return (values * repeats)[: self.embedding_dim]

    @staticmethod
    def _clamp(v: float) -> float:
        return max(0.0, min(10.0, v))

    def infer(self, chunk_text: str) -> dict:
        words = chunk_text.split()
        word_count = len(words)
        pace_score = self._clamp(4.0 + min(word_count, 50) * 0.08)
        prosody_score = self._clamp(
            5.0 + (1.0 if any(c in "?!" for c in chunk_text) else 0.0)
                + (0.5 if len(set(words)) / max(1, word_count) > 0.6 else 0.0)
        )
        return {
            "embedding":  self._hash_to_vector(f"audio::{chunk_text[:80]}"),
            "voice_pace": pace_score,
            "prosody":    prosody_score,
        }


print("Encoders defined")


✅ Encoders defined


## 3 · Cross-Attention Fusion & Hierarchical Scoring Heads

In [4]:
class FusionHead:
    """
    Cross-attention fusion layer.
    Computes per-modality attention weights from embedding energy and
    produces a fused vector. Mirrors the Cross-Attention Transformer
    block in the architecture diagram.
    """

    @staticmethod
    def _mean(values: list[float]) -> float:
        return sum(values) / max(1, len(values))

    def infer(self, text_emb: list[float], visual_emb: list[float], audio_emb: list[float]) -> dict:
        te = self._mean(text_emb)
        ve = self._mean(visual_emb)
        ae = self._mean(audio_emb)

        total = max(1e-6, te + ve + ae)
        tw, vw, aw = te / total, ve / total, ae / total

        fused = [(tw * t) + (vw * v) + (aw * a)
                 for t, v, a in zip(text_emb, visual_emb, audio_emb)]

        return {
            "vector": fused,
            "attention": {
                "text":   round(tw, 4),
                "visual": round(vw, 4),
                "audio":  round(aw, 4),
            },
        }


class ScoringHead:
    """
    Multi-output regression scoring head.
    6 text-focused metrics (scores 1-6) + 4 AV-focused metrics (scores 7-10).
    Weighted aggregation: text 50 % · AV 35 % · fusion signal 15 %.
    """

    @staticmethod
    def _avg(values: list[float]) -> float:
        return sum(values) / max(1, len(values))

    @staticmethod
    def _clamp(v: float) -> float:
        return max(0.0, min(10.0, v))

    def infer(self, text_f: dict, visual_f: dict, audio_f: dict, fused: dict) -> dict:
        text_scores = {
            "Problem Clarity":         float(text_f["problem_clarity"]),
            "Market Opportunity":      float(text_f["market_opportunity"]),
            "Solution Uniqueness":     float(text_f["solution_uniqueness"]),
            "Traction Evidence":       float(text_f["traction_evidence"]),
            "Business Model Strength": float(text_f["business_model_strength"]),
            "Team Readiness":          float(text_f["team_readiness"]),
        }
        av_scores = {
            "Delivery Clarity":     float(visual_f["delivery_clarity"]),
            "Presenter Confidence": float(visual_f["presenter_confidence"]),
            "Voice Pace":           float(audio_f["voice_pace"]),
            "Voice Prosody":        float(audio_f["prosody"]),
        }

        text_avg     = self._avg(list(text_scores.values()))
        av_avg       = self._avg(list(av_scores.values()))
        fusion_signal = self._avg(fused["vector"]) * 10.0

        aggregate  = self._clamp(text_avg * 0.50 + av_avg * 0.35 + fusion_signal * 0.15)
        confidence = self._clamp(av_avg  * 0.50 + text_avg * 0.50)

        return {
            "text_scores": text_scores,
            "av_scores":   av_scores,
            "aggregate":   round(aggregate, 2),
            "confidence":  round(confidence, 2),
        }


print("Fusion & Scoring heads defined")


✅ Fusion & Scoring heads defined


## 4 · Risk Detection & Automated Feedback Engine

In [5]:
def detect_risk_flags(chunk_text: str, aggregate_score: float) -> list[str]:
    """Heuristic risk flag detection per chunk."""
    low = chunk_text.lower()
    flags = []
    if aggregate_score < 5.5:
        flags.append("low-quality-signal")
    if "no competition" in low or "no competitor" in low:
        flags.append("competitive-landscape-missing")
    if "guaranteed" in low or "100%" in low:
        flags.append("overclaim-risk")
    if "soon" in low and "revenue" in low:
        flags.append("traction-evidence-weak")
    return flags


def build_feedback(overall_score: float, text_avg: float, av_avg: float,
                   top_risks: list[str]) -> dict:
    """Generate automated strengths, weaknesses, and suggestions."""
    strengths, weaknesses, suggestions = [], [], []

    if text_avg >= 7:
        strengths.append("Strong articulation of problem and market context")
    else:
        weaknesses.append("Narrative around problem-market fit needs sharper framing")
        suggestions.append("Open with customer pain backed by one concrete metric")

    if av_avg >= 7:
        strengths.append("Confident delivery and visual support")
    else:
        weaknesses.append("Presentation confidence and pacing can improve")
        suggestions.append("Tighten slide text and slow down key transitions")

    if overall_score >= 8:
        strengths.append("Investor readiness appears high")
    elif overall_score < 6:
        suggestions.append("Add traction proof points and a clearer business model story")

    if "competitive-landscape-missing" in top_risks:
        weaknesses.append("Competitive positioning is not clearly defended")
        suggestions.append("Add competitor matrix with clear differentiation")

    if "overclaim-risk" in top_risks:
        weaknesses.append("Claims may appear inflated to investors")
        suggestions.append("Replace absolute claims with verifiable evidence")

    return {"strengths": strengths, "weaknesses": weaknesses, "suggestions": suggestions}


def build_investor_dashboard(quantitative_scores: list[dict],
                              modality_attention: dict[str, float],
                              risk_counts: dict[str, int]) -> dict:
    """Build chart-ready payload for the Investor Dashboard (React/MERN)."""
    return {
        "quantitative_scores": [
            {"label": item["name"], "value": round(item["score"], 2)}
            for item in quantitative_scores
        ],
        "modality_weights": [
            {"label": k, "value": round(v, 2)} for k, v in modality_attention.items()
        ],
        "risk_distribution": [
            {"label": k, "value": float(v)} for k, v in sorted(risk_counts.items())
        ],
    }


print("Risk detection & reporting defined")


✅ Risk detection & reporting defined


## 5 · Temporal Synchronizer & Segmenter
Splits the pitch into 5-second aligned chunks across all modalities.

In [6]:
from dataclasses import dataclass


@dataclass
class PitchChunk:
    chunk_id: int
    start_sec: int
    end_sec: int
    text: str
    slide_context: str


def temporal_synchronize_and_segment(payload: PitchInput,
                                      window_seconds: int = 5) -> list[PitchChunk]:
    """
    Sliding-window segmenter (5 s default).
    Distributes transcript words proportionally across timeline chunks
    and attaches the resolved slide context to every chunk.
    """
    # Resolve transcript
    transcript = payload.transcript.strip()
    if not transcript and payload.video and payload.video.transcript_text.strip():
        transcript = payload.video.transcript_text.strip()

    # Resolve slide context
    if payload.slides:
        slide_context = " ".join(
            f"{s.title} {s.content}".strip() for s in payload.slides
        ).strip()
    else:
        slide_context = " ".join(payload.slide_text).strip()

    duration = payload.video.duration_sec if payload.video else 60
    num_chunks = max(1, (duration + window_seconds - 1) // window_seconds)

    sentences = [s.strip() for s in transcript.replace("\n", " ").split(".") if s.strip()]
    if not sentences:
        sentences = [transcript or "No transcript available"]
    words = " ".join(sentences).split()
    words_per_chunk = max(1, len(words) // num_chunks) if words else 1

    chunks = []
    for i in range(num_chunks):
        start = i * window_seconds
        end = min((i + 1) * window_seconds, duration)
        w_start = i * words_per_chunk
        w_end = min((i + 1) * words_per_chunk, len(words))
        chunk_text = " ".join(words[w_start:w_end]) if w_start < len(words) else                      sentences[min(i, len(sentences) - 1)]
        chunks.append(PitchChunk(
            chunk_id=i, start_sec=start, end_sec=end,
            text=chunk_text, slide_context=slide_context,
        ))

    return chunks


print(f"Preprocessor defined")


✅ Preprocessor defined


## 6 · Full Evaluation Pipeline
Orchestrates all stages end-to-end.

In [7]:
_TEXT_RATIONALES = {
    "Problem Clarity":         "How clearly the startup pain point is communicated.",
    "Market Opportunity":      "Presence and depth of market framing.",
    "Solution Uniqueness":     "Distinctiveness of the proposed solution.",
    "Traction Evidence":       "Evidence of adoption, growth, pilots, or revenue.",
    "Business Model Strength": "Clarity on monetization and economics.",
    "Team Readiness":          "Signals of execution capability and preparedness.",
}
_AV_RATIONALES = {
    "Delivery Clarity":     "Quality of visual communication and structure.",
    "Presenter Confidence": "Confidence cues from visual stream proxies.",
    "Voice Pace":           "Speech pacing suitability for investor comprehension.",
    "Voice Prosody":        "Variation and emphasis in delivery.",
}


class StartupPitchPipeline:
    """
    End-to-end multimodal evaluation pipeline.

    Stages:
    1. Temporal segmentation (5 s chunks)
    2. Parallel feature extraction: Text (NLP) · Visual (CV) · Audio (DSP)
    3. Cross-attention fusion
    4. Hierarchical scoring (10 metrics)
    5. Risk flag detection
    6. Automated feedback + investor dashboard
    """

    def __init__(self, window_seconds: int = 5):
        self.window_seconds = window_seconds
        self.text_enc   = TextEncoder()
        self.visual_enc = VisualEncoder()
        self.audio_enc  = AudioEncoder()
        self.fusion     = FusionHead()
        self.scorer     = ScoringHead()

    def evaluate(self, payload: PitchInput, request_id: str = "eval-001") -> EvaluationResponse:
        chunks = temporal_synchronize_and_segment(payload, self.window_seconds)
        user_stage = payload.user_details.stage if payload.user_details else ""

        chunk_reports: list[ChunkReport] = []
        agg_scores, conf_scores, text_scores, av_scores = [], [], [], []
        lang_preds: list[str] = []
        all_quant: list[dict] = []
        modality_totals = {"text": 0.0, "visual": 0.0, "audio": 0.0}
        risk_counts: dict[str, int] = {}

        for chunk in chunks:
            tf = self.text_enc.infer(chunk.text, payload.language_hint, chunk.slide_context)
            vf = self.visual_enc.infer(chunk.slide_context, chunk.chunk_id, user_stage)
            af = self.audio_enc.infer(chunk.text)
            lang_preds.append(tf["language_detected"])

            fused  = self.fusion.infer(tf["embedding"], vf["embedding"], af["embedding"])
            scored = self.scorer.infer(tf, vf, af, fused)

            text_metrics = [MetricScore(name=n, score=round(s, 2), rationale=_TEXT_RATIONALES[n])
                            for n, s in scored["text_scores"].items()]
            av_metrics   = [MetricScore(name=n, score=round(s, 2), rationale=_AV_RATIONALES[n])
                            for n, s in scored["av_scores"].items()]

            all_quant.extend({"name": m.name, "score": m.score} for m in text_metrics + av_metrics)
            modality_totals["text"]   += fused["attention"]["text"]
            modality_totals["visual"] += fused["attention"]["visual"]
            modality_totals["audio"]  += fused["attention"]["audio"]

            t_avg = sum(m.score for m in text_metrics) / len(text_metrics)
            a_avg = sum(m.score for m in av_metrics)   / len(av_metrics)
            text_scores.append(t_avg)
            av_scores.append(a_avg)
            agg_scores.append(scored["aggregate"])
            conf_scores.append(scored["confidence"])

            flags = detect_risk_flags(chunk.text, scored["aggregate"])
            for f in flags:
                risk_counts[f] = risk_counts.get(f, 0) + 1

            chunk_reports.append(ChunkReport(
                chunk_id=chunk.chunk_id, start_sec=chunk.start_sec, end_sec=chunk.end_sec,
                text_metrics=text_metrics, av_metrics=av_metrics,
                attention=fused["attention"], risk_flags=flags,
                aggregate_score=scored["aggregate"],
            ))

        n = max(1, len(chunks))
        overall_score  = round(sum(agg_scores)  / n, 2)
        conf_score     = round(sum(conf_scores) / n, 2)
        t_avg_overall  = sum(text_scores) / n
        a_avg_overall  = sum(av_scores)   / n

        sorted_risks = [r for r, _ in sorted(risk_counts.items(), key=lambda x: x[1], reverse=True)]
        feedback = build_feedback(overall_score, t_avg_overall, a_avg_overall, sorted_risks)

        investment_band = (
            "high-potential" if overall_score >= 8.0 else
            "watchlist"      if overall_score >= 6.0 else
            "early-risk"
        )
        language_detected = max(set(lang_preds), key=lang_preds.count)

        avg_modalities = {k: round(v / n, 4) for k, v in modality_totals.items()}

        # Average per metric across chunks
        metric_agg: dict[str, list[float]] = {}
        for m in all_quant:
            metric_agg.setdefault(m["name"], []).append(m["score"])
        metric_rows = [{"name": k, "score": sum(v) / len(v)} for k, v in metric_agg.items()]

        dash_raw = build_investor_dashboard(metric_rows, avg_modalities, risk_counts)
        dashboard = InvestorDashboard(
            quantitative_scores=[DashboardSeriesPoint(**p) for p in dash_raw["quantitative_scores"]],
            modality_weights   =[DashboardSeriesPoint(**p) for p in dash_raw["modality_weights"]],
            risk_distribution  =[DashboardSeriesPoint(**p) for p in dash_raw["risk_distribution"]],
        )

        summary = EvaluationSummary(
            overall_score=overall_score, confidence_score=conf_score,
            investment_band=investment_band, language_detected=language_detected,
            **feedback,
        )

        return EvaluationResponse(
            request_id=request_id,
            summary=summary,
            chunk_reports=chunk_reports,
            dashboard=dashboard,
        )


print("Pipeline defined")


✅ Pipeline defined


## 7 · 📂 Dataset Upload  *(leave blank — add your data here)*

When your dataset is ready, upload pitch data using one of these patterns:

```python
# Option A — Google Drive mount
from google.colab import drive
drive.mount('/content/drive')
# dataset_path = '/content/drive/MyDrive/your_dataset.csv'

# Option B — direct file upload
from google.colab import files
uploaded = files.upload()

# Option C — define pitches inline (see Section 8 for the sample format)
```

**Expected fields per pitch:** `title`, `transcript`, `language_hint`, `slides`, `video`, `user_details`


In [8]:
# ── Paste your dataset loading code here ──────────────────────────────────
# dataset = ...
# pitches = [PitchInput(**row) for row in dataset]

print("Dataset section — add your loading code above, then run Section 8.")


⏳ Dataset section — add your loading code above, then run Section 8.


## 8 · Demo Evaluation
Runs the full pipeline on a sample pitch to verify everything works end-to-end.

In [17]:
import json

sample_pitch = PitchInput(
    title="AgriSage",
    transcript="",
    language_hint="en-ta",
    video=PitchVideoInput(
        file_name="agrisage_pitch.mp4",
        duration_sec=120,
        transcript_text=(
            "We solve crop loss for farmers using bilingual AI support. "
            "Our pilot improved yield and market access in Tamil Nadu. "
            "The market is large with underserved smallholder farmers. "
            "Our subscription model generates recurring revenue from cooperatives."
        ),
    ),
    slides=[
        SlideInput(title="Problem",      content="Post-harvest crop loss is high"),
        SlideInput(title="Market",       content="Large underserved smallholder segment"),
        SlideInput(title="Solution",     content="AI-powered bilingual advisory platform"),
        SlideInput(title="Traction",     content="Pilot growth and repeat usage"),
        SlideInput(title="Business Model", content="SaaS subscription for cooperatives"),
    ],
    user_details=UserDetails(
        founder_name="Aravind",
        startup_name="AgriSage",
        sector="AgriTech",
        stage="Seed",
    ),
)

pipeline = StartupPitchPipeline(window_seconds=5)
result   = pipeline.evaluate(sample_pitch, request_id="demo-agrisage")

print("=" * 60)
print(f"  Startup      : {sample_pitch.title}")
print(f"  Request ID   : {result.request_id}")
print(f"  Overall Score: {result.summary.overall_score} / 10")
print(f"  Confidence   : {result.summary.confidence_score} / 10")
print(f"  Investment   : {result.summary.investment_band.upper()}")
print(f"  Language     : {result.summary.language_detected}")
print(f"  Chunks       : {len(result.chunk_reports)}")
print("=" * 60)
print("\nStrengths:")
for s in result.summary.strengths:
    print(f"{s}")
print("\nWeaknesses:")
for w in result.summary.weaknesses:
    print(f"{w}")
print("\nSuggestions:")
for s in result.summary.suggestions:
    print(f"{s}")


  Startup      : AgriSage
  Request ID   : demo-agrisage
  Overall Score: 5.09 / 10
  Confidence   : 5.12 / 10
  Investment   : EARLY-RISK
  Language     : ta-en
  Chunks       : 24

Strengths:

Weaknesses:
Narrative around problem-market fit needs sharper framing
Presentation confidence and pacing can improve

Suggestions:
Open with customer pain backed by one concrete metric
Tighten slide text and slow down key transitions
Add traction proof points and a clearer business model story


## 9 · Quantitative Scores (10 Metrics)

In [15]:
print(f"{'Metric':<30} {'Score':>6}")
print("-" * 38)
for pt in result.dashboard.quantitative_scores:
    print(f"{pt.label:<30} {pt.value:>5.2f}")


Metric                          Score
--------------------------------------
Problem Clarity                 5.50
Market Opportunity              5.47
Solution Uniqueness             7.50
Traction Evidence               3.66
Business Model Strength         3.57
Team Readiness                  4.00
Delivery Clarity                5.09
Presenter Confidence            6.52
Voice Pace                      4.08
Voice Prosody                   5.50


## 10 · Modality Attention Weights

In [16]:
print("\nModality Attention Weights:")
for pt in result.dashboard.modality_weights:
    pct = pt.value * 100
    print(f"  {pt.label:<8} {pct:5.1f}%")



Modality Attention Weights:
  text      33.0%
  visual    34.0%
  audio     33.0%


## 11 · Per-Chunk Reports

In [12]:
print(f"\nDetailed chunk reports ({len(result.chunk_reports)} chunks):\n")
for cr in result.chunk_reports:
    print(f"  Chunk {cr.chunk_id:02d}  [{cr.start_sec:3d}s – {cr.end_sec:3d}s]  "
          f"score={cr.aggregate_score:.2f}  "
          f"risks={cr.risk_flags or 'none'}")



Detailed chunk reports (24 chunks):

  Chunk 00  [  0s –   5s]  score=4.85  risks=['low-quality-signal']
  Chunk 01  [  5s –  10s]  score=5.03  risks=['low-quality-signal']
  Chunk 02  [ 10s –  15s]  score=4.93  risks=['low-quality-signal']
  Chunk 03  [ 15s –  20s]  score=5.08  risks=['low-quality-signal']
  Chunk 04  [ 20s –  25s]  score=5.26  risks=['low-quality-signal']
  Chunk 05  [ 25s –  30s]  score=5.02  risks=['low-quality-signal']
  Chunk 06  [ 30s –  35s]  score=4.97  risks=['low-quality-signal']
  Chunk 07  [ 35s –  40s]  score=4.99  risks=['low-quality-signal']
  Chunk 08  [ 40s –  45s]  score=5.20  risks=['low-quality-signal']
  Chunk 09  [ 45s –  50s]  score=5.19  risks=['low-quality-signal']
  Chunk 10  [ 50s –  55s]  score=4.88  risks=['low-quality-signal']
  Chunk 11  [ 55s –  60s]  score=5.26  risks=['low-quality-signal']
  Chunk 12  [ 60s –  65s]  score=5.15  risks=['low-quality-signal']
  Chunk 13  [ 65s –  70s]  score=5.12  risks=['low-quality-signal']
  Chunk 14

## 12 · Batch Evaluation
Evaluate multiple pitches at once.

In [13]:
def evaluate_batch(pitches: list[PitchInput],
                   pipeline: StartupPitchPipeline) -> list[EvaluationResponse]:
    """Evaluate a list of pitches and return all responses."""
    if not pitches:
        raise ValueError("pitches list cannot be empty")
    return [pipeline.evaluate(p, request_id=f"batch-{i}") for i, p in enumerate(pitches)]


# Example batch (add more pitches from your dataset in Section 7)
batch_pitches = [
    PitchInput(
        title="EdTech Tamil",
        transcript="We help students learn coding in Tamil. "
                   "Our platform has 500 active users and revenue growing 20% month-on-month.",
        language_hint="en-ta",
        video=PitchVideoInput(file_name="edtech.mp4", duration_sec=60),
        slides=[SlideInput(title="Problem", content="Language barrier in coding education")],
        user_details=UserDetails(startup_name="TamilCode", stage="Seed"),
    ),
    PitchInput(
        title="HealthAI",
        transcript="AI diagnostics for rural clinics. No competition in this space. Revenue coming soon.",
        language_hint="en",
        video=PitchVideoInput(file_name="health.mp4", duration_sec=90),
        slides=[SlideInput(title="Market", content="Huge rural healthcare gap")],
        user_details=UserDetails(startup_name="HealthAI", stage="Pre-seed"),
    ),
]

batch_results = evaluate_batch(batch_pitches, pipeline)

print(f"Batch evaluation complete: {len(batch_results)} pitches\n")
print(f"{'Title':<20} {'Score':>6}  {'Band':<16}  {'Risks'}")
print("-" * 65)
for res, pitch in zip(batch_results, batch_pitches):
    s = res.summary
    all_risks = set()
    for cr in res.chunk_reports:
        all_risks.update(cr.risk_flags)
    print(f"{pitch.title:<20} {s.overall_score:>6.2f}  {s.investment_band:<16}  {list(all_risks) or 'none'}")


Batch evaluation complete: 2 pitches

Title                 Score  Band              Risks
-----------------------------------------------------------------
EdTech Tamil           4.98  early-risk        ['low-quality-signal']
HealthAI               4.95  early-risk        ['traction-evidence-weak', 'low-quality-signal']


## 13 · Quick Self-Tests

In [14]:
def run_tests():
    p = pipeline

    # Test 1: basic output shape
    r = p.evaluate(sample_pitch, "t1")
    assert 0 <= r.summary.overall_score <= 10
    assert 0 <= r.summary.confidence_score <= 10
    assert r.summary.investment_band in {"high-potential", "watchlist", "early-risk"}
    assert r.summary.language_detected in {"en", "ta", "ta-en"}
    assert len(r.chunk_reports) > 0
    assert len(r.dashboard.quantitative_scores) == 10
    print("Test 1 passed: output shapes correct")

    # Test 2: Tamil-English detection
    mixed = PitchInput(
        title="Mixed",
        transcript="This startup helps farmers with தமிழ் சந்தை access.",
        language_hint="en-ta",
        video=PitchVideoInput(file_name="mix.mp4", duration_sec=10),
    )
    r2 = p.evaluate(mixed, "t2")
    assert r2.summary.language_detected == "ta-en", r2.summary.language_detected
    print("Test 2 passed: Tamil-English detection works")

    # Test 3: risk flag detection
    risky = PitchInput(
        title="Risky",
        transcript="We are guaranteed to succeed. No competitor exists. Revenue coming soon.",
        language_hint="en",
        video=PitchVideoInput(file_name="risky.mp4", duration_sec=10),
    )
    r3 = p.evaluate(risky, "t3")
    all_flags = {f for cr in r3.chunk_reports for f in cr.risk_flags}
    assert "overclaim-risk" in all_flags, all_flags
    assert "competitive-landscape-missing" in all_flags, all_flags
    assert "traction-evidence-weak" in all_flags, all_flags
    print("Test 3 passed: risk flags detected correctly")

    # Test 4: short video produces exactly 1 chunk
    short = PitchInput(
        title="Short",
        transcript="Quick pitch.",
        language_hint="en",
        video=PitchVideoInput(file_name="short.mp4", duration_sec=5),
    )
    r4 = p.evaluate(short, "t4")
    assert len(r4.chunk_reports) == 1
    print("Test 4 passed: single chunk for 5-second video")

    print("\nAll tests passed!")

run_tests()


✅ Test 1 passed: output shapes correct
✅ Test 2 passed: Tamil-English detection works


AssertionError: {'low-quality-signal', 'competitive-landscape-missing', 'overclaim-risk'}